# Least Squares Fitting

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

## Defining system model
### Step response equation
The system is modeled as a third order LTI system with three distict time constants.
$$
c(t) = K ( 1 - Ae^{-t/\tau_1} - Be^{-t/\tau_2} - Ce^{-t/\tau_3} )
$$
Ideally: A + B + C = 1 for K.

In [36]:
def step_response_model(t, K, A, tau1, B, tau2, C, tau3):
    """
    Model for a step response with 3 time constants
    """
    return K * (1 - A * np.exp(-t / tau1) - B * np.exp(-t / tau2) - C * np.exp(-t / tau3))

## Loading the extracted data from the CSVs
Since the csv for the range 0 to 5 seconds it's just a low-res version of the other two, 
ignore it.

Combine the fast part of the curve (0 to 1 sec) and the slow part of the curve (1 to 5 sec) for precise data.

In [37]:
df_fast = pd.read_csv('csv/0to1.csv')
df_slow = pd.read_csv('csv/1to5.csv')
df_final = pd.concat([df_fast, df_slow])
db_final = df_final.sort_values(by='t').drop_duplicates(subset=['t'])
# Drop possible negative values from the extraction
df_final = df_final[df_final['t'] >= 0]

Extract arrays for the optimizer

In [38]:
t_data = df_final['t'].values
v_data = df_final['v'].values

print(f"Total data points: {len(t_data)}")

Total data points: 139


### Initial Guesses
Based from the provided images,
$$
\lim_{t \to 5} c(t) \approx 4.0 \text{ V}
$$

Using the Final Value Theorem,
$$
\lim_{t \to \infty} c(t) = \lim_{t \to \infty} K ( 1 - A e^{-t/\tau_1} - B e^{-t/\tau_2} - C e^{-t/\tau_3} )
$$
$$
c(\infty) = K (1 - 0 - 0 - 0)
$$
$$
c(\infty) = K
$$
Since the system appears to settle at $y=4$ on the graph,
$$
c(\infty) = 4
$$
$$
K = 4
$$

Initial Guesses (p0): [K, A, tau1, B, tau2, C, tau3]
$$ K \approx 4 $$

From 0 to 0.1 sec, the graph shoots up immediately (0 to 0.8V)
$$\tau_1 \approx 0.1 $$

From 0.1 to 0.8, the change in slope of the curve decreases but is still significant
$$\tau_2 \approx 0.5 $$

From 0.8 to 5, the change is a slow and gradual climb
$$\tau_2 \approx 2.0 $$

For $c(0) = K (1 - A - B - C) = 0)$ to hold true, the weights $ A + B + C = 1.0 $
$$ A \approx 0.33 $$
$$ B \approx 0.33 $$
$$ C \approx 0.33 $$


In [39]:
p0_guess = [4.0, 0.333, 0.1, 0.333, 0.5, 0.334, 2.0]

### Finding the values using Non-Linear Least Squares optimization (Levenberg-Marquardt algorithm)

In [40]:
try:
    popt, pcov = curve_fit(step_response_model, t_data, v_data, p0=p0_guess, maxfev=20000)
    
    K, A, t1, B, t2, C, t3 = popt
    print(f"OPTIMIZED PARAMETERS:")
    print(f"Steady State Gain (K) = {K:.4f}")
    print(f"Time Constants: τ1={t1:.4f}s, τ2={t2:.4f}s, τ3={t3:.4f}s")
    
except Exception as e:
    print(f"Curve fitting failed: {e}")
    print("Need to adjust the 'p0_guess' values closer to the graph.")

OPTIMIZED PARAMETERS:
Steady State Gain (K) = 4.0004
Time Constants: τ1=0.0941s, τ2=0.3225s, τ3=0.9951s


# Identification & Visualization - Hugh